In [1]:
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.38 langchain-groq==0.1.9 faiss-cpu sentence-transformers langchain-huggingface pypdf duckduckgo-search

In [12]:
import os
from langchain_groq import ChatGroq

#api key is set in streamlit settings
os.environ["GROQ_API_KEY"] = ""

# Create LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("LLM ready!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

LLM ready!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [4]:
import pandas as pd

# Create sample employee data
data = {
    "employee_id": [101, 102, 103, 104, 105],
    "name": ["Sowmya", "Rahul", "Priya", "Arjun", "Sneha"],
    "department": ["Engineering", "HR", "Engineering", "Sales", "HR"],
    "leave_balance": [12, 8, 5, 15, 3],
    "leaves_taken": [3, 7, 10, 0, 12]
}

df = pd.DataFrame(data)
df.to_csv("employee_data.csv", index=False)
print(df)
print("CSV created!")

   employee_id    name   department  leave_balance  leaves_taken
0          101  Sowmya  Engineering             12             3
1          102   Rahul           HR              8             7
2          103   Priya  Engineering              5            10
3          104   Arjun        Sales             15             0
4          105   Sneha           HR              3            12
CSV created!


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load PDF
loader = PyPDFLoader("07_pay_benefits_and_leave_policy.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# Create embeddings and store in FAISS
embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print("FAISS vectorstore ready!")

Loaded 11 pages
Created 52 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vectorstore ready!


In [6]:
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1 - PDF Search
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(query):
    docs = retriever.get_relevant_documents(query)
    return "\n".join([doc.page_content for doc in docs])

pdf_tool = Tool(
    name="HR Policy Search",
    func=search_hr_policy,
    description="Use this for questions about HR policies, leave policy, salary, work from home rules."
)

# Tool 2 - CSV Search
df = pd.read_csv("employee_data.csv")

def search_employee_data(query):
    return f"Employee Data:\n{df.to_string()}\n\nQuestion: {query}"

csv_tool = Tool(
    name="Employee Data Search",
    func=search_employee_data,
    description="Use this for questions about employee leave balance, department, leaves taken."
)

# Tool 3 - Web Search
search = DuckDuckGoSearchRun()

web_tool = Tool(
    name="Web Search",
    func=search.run,
    description="Use this for current news or general questions not related to HR or employee data."
)

print("All 3 tools created!")

All 3 tools created!


In [7]:
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

# Combine all tools
tools = [pdf_tool, csv_tool, web_tool]

# Add memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

print("Agent ready!")

Agent ready!


/tmp/ipykernel_1717/3470468463.py:14: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [8]:
# Test 1 - HR Policy question
response = agent.run("What is the leave policy?")
print(response)

/tmp/ipykernel_1717/408135844.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  response = agent.run("What is the leave policy?")




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: HR Policy Search
Action Input: leave policy
Observation: 10 
 
Employees must ensure that:  
 
• leave is authorized,  
• leave taken is recorded on a timely basis,  
• leave balance records are correct, and  
• leave banks are not exceeded. 
 
Paid Absence Prior to Retirement  
Once an employee commences the use of earned, purchased or accumulated time 
immediately prior to retirement, the normal employment relationship is considered to be 
modified in that the employee is no longer committed to return to work and, likewise, the
1 
 
Human Resources Policy 07 – Pay, Benefits and Leave  
Last updated: August 17, 2018 
 
This policy statement covers pay, benefits and leave for both excluded and bargaining unit 
employees appointed under the Public Service Act. The policy statement supports the core 
policy objective of ensuring that “government is supported by a professional public service 
that has th

/tmp/ipykernel_1717/2127165144.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)


Thought: Do I need to use a tool? No
AI: Based on the information provided, it appears that the leave policy is quite detailed and covers various aspects such as leave authorization, recording, balance records, and paid absence prior to retirement. It's essential for employees to understand and adhere to these policies to ensure a smooth leave process. If you have any specific questions or concerns regarding the leave policy, I'd be happy to help clarify them.

> Finished chain.
Based on the information provided, it appears that the leave policy is quite detailed and covers various aspects such as leave authorization, recording, balance records, and paid absence prior to retirement. It's essential for employees to understand and adhere to these policies to ensure a smooth leave process. If you have any specific questions or concerns regarding the leave policy, I'd be happy to help clarify them.


In [9]:
# Test 2 - Employee data
response = agent.run("How many leave balance does employee Sowmya have in the employee database?")
print(response)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Employee Data Search
Action Input: Sowmya's leave balance
Observation: Employee Data:
   employee_id    name   department  leave_balance  leaves_taken
0          101  Sowmya  Engineering             12             3
1          102   Rahul           HR              8             7
2          103   Priya  Engineering              5            10
3          104   Arjun        Sales             15             0
4          105   Sneha           HR              3            12

Question: Sowmya's leave balance
Thought:Thought: Do I need to use a tool? No
AI: Sowmya has 12 days of leave balance.

> Finished chain.
Sowmya has 12 days of leave balance.


In [11]:
# Test 3 - Force web search
response = agent.run("Search the web and tell me the latest news about artificial intelligence today")
print(response)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Web Search
Action Input: latest news about artificial intelligence today

/usr/local/lib/python3.12/dist-packages/langchain_community/utilities/duckduckgo_search.py:61: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/langchain_community/utilities/duckduckgo_search.py:62: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.


Observation: No good DuckDuckGo Search Result was found
Thought:

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Thought: Do I need to use a tool? Yes
Action: Web Search
Action Input: latest news about artificial intelligence today

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Observation: Latest Current News: U.S., World, Entertainment, Health, Business, Technology, Politics, Sports. View the latest news and breaking news today for U.S., world, weather, entertainment, politics and health at CNN.com. Learn about the latest news events happening around the nation with Fox News. Go to NBCNews.com for breaking news, videos, and the latest top stories in world news, business, politics, health and pop culture. Stay informed with a redesigned home feed, personalized content, and a 24/7 live news stream. The doubles tournament begins July …
Thought:

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Thought: Do I need to use a tool? No
AI: It seems that I was unable to find any specific news about artificial intelligence through my previous web searches. However, I can suggest some reputable news sources that you can check for the latest news on AI, such as CNN, Fox News, and NBC News. They often have dedicated sections for technology and AI news. Additionally, you can try searching for AI-related news on other popular news websites or searching for specific AI-related topics on the web. I'll do my best to provide you with more general information on AI if you'd like.

> Finished chain.
It seems that I was unable to find any specific news about artificial intelligence through my previous web searches. However, I can suggest some reputable news sources that you can check for the latest news on AI, such as CNN, Fox News, and NBC News. They often have dedicated sections for technology and AI news. Additionally, you can try searching for AI-related news on other popular news websites 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/langchain_groq/chat_models.py:665: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  response = response.dict()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-package